# A (Very Short) Introduction to Machine Learning

In this course, we will study **Gradient Descent** (arguably the most widely known and employed optimization method), **Stochastic Gradient Descent** (its most common variant) and **Adam** (the de-facto standard optimizer for modern Deep Learning). To make the discussion concrete, we will apply all of them to the problem of **training a Neural Network**, which we will build *from scratch* using only `numpy`.

Before doing that, we need a common language. In this first chapter, we introduce the fundamental concepts of Artificial Intelligence and Machine Learning from a mathematical perspective. In particular, we will discuss what we mean by *learning*, how the learning problem can be mathematically formulated, which are the most common types of learning problems, and we will outline the typical pipeline used to understand, design, and solve a Machine Learning problem.

The course is organized in five chapters:

1. **Introduction to Machine Learning** (this chapter): what are AI, ML, DL and Data Science, and how a learning problem is formulated.
2. **Neural Networks**: how to build a classifier as a graph of simple computational units, up to the Multi-Layer Perceptron (MLP), and how to compute its gradient via *backpropagation*.
3. **Gradient Descent**: the mathematics of optimization (convexity, optimality conditions) and the Gradient Descent algorithm.
4. **Stochastic Gradient Descent**: how to deal with datasets with a large number of samples, using mini-batches.
5. **Adam and a realistic experiment**: adaptive optimizers and the full training of a Neural Network on a real dataset.

## AI, Machine Learning, Deep Learning and Data Science

Nowadays, the terms *Artificial Intelligence*, *Machine Learning*, *Deep Learning* and *Data Science* are used almost interchangeably in newspapers and social media. However, they refer to (related but) different concepts, and it is worth fixing their meaning once and for all.

- **Artificial Intelligence (AI)** is the broadest of these terms. It refers to the collection of techniques that allow a machine to perform tasks that are usually associated with human intelligence, such as reasoning, planning, perceiving the environment, understanding language and taking decisions. Note that this definition does **not** say anything about *how* such a behavior is obtained: a chess engine based on a set of hand-written rules and a tree search is an AI system, even if it never *learned* anything from data.

- **Machine Learning (ML)** is the subset of AI in which the intelligent behavior is not explicitly programmed, but it is **learned from data**. Instead of writing the rules by hand, we write an algorithm that *extracts* the rules from a (large) set of examples.

- **Deep Learning (DL)** is the subset of Machine Learning based on **(deep) Neural Networks**, i.e. models obtained by composing a large number of simple, parametric, non-linear functions (the *layers*). Essentially all the recent breakthroughs in AI (image recognition, machine translation, Large Language Models such as ChatGPT or Claude, image generation models, ...) are based on Deep Learning.

- **Data Science (DS)** is a more *transversal* discipline, whose aim is to extract knowledge and insights from data. It uses tools from Statistics, Computer Science and (of course) Machine Learning, but it also includes tasks such as data collection, data cleaning, visualization and communication of the results. Thus, Data Science partially overlaps with ML, but it is not contained in AI.

The relationship among these fields is summarized in the following diagram.

![AI contains Machine Learning and Deep Learning; Data Science overlaps with them.](./fig/ai_ml_dl_data_science.png)

In this course, we will mostly focus on Machine Learning and, at the end, on the simplest example of Deep Learning model. Let's start with a widely accepted definition of **Machine Learning**:

> **Machine Learning (ML)** is the collection of techniques and algorithms that extract knowledge from data and use that knowledge to make accurate predictions.

## The learning problem

A learning problem involves extracting knowledge from a dataset of $N$ points:

$$
\mathcal{D} = \{ (x^{(1)}, y^{(1)}), (x^{(2)}, y^{(2)}), \dots, (x^{(N)}, y^{(N)}) \}
$$

where each pair $(x^{(i)}, y^{(i)})$ consists of an input and its corresponding output. Think of the input data $x^{(i)} \in \mathbb{R}^d$ as the available information and the output data $y^{(i)} \in \mathbb{R}^s$ as the quantity we want to predict. The input dimension, $d$, represents the number of input **features**, while the output dimension, $s$, corresponds to the number of variables we aim to predict.

For example, Machine Learning is widely used to predict house prices based on property characteristics. Suppose we have a dataset where each house is described by features such as:

- $x_1$: Square footage
- $x_2$: Number of bedrooms
- $x_3$: Number of bathrooms
- $x_4$: Distance to the city center
- $x_5$: Age of the house

The goal is to predict the selling price $y$ of a house given its features. Here, the input $x^{(i)} \in \mathbb{R}^5$ represents a vector of property attributes, and the output $y^{(i)} \in \mathbb{R}$ is the house price.

A Machine Learning model learns a function $f_\Theta$ that maps inputs to outputs:

$$
y \approx f_\Theta(x_1, x_2, x_3, x_4, x_5).
$$

Initially, the model **does not know** how these features influence the price. By **training** on past sales data, it finds patterns and estimates parameters (or *weights*) $\Theta$ that best fit the data. Once trained, the model can predict the price of new houses that were not in the dataset.

This simple example illustrates the core idea of learning: using past data to make future predictions. More generally, a Machine Learning model can be seen as a mathematical function that maps input features to an output of interest.

```{warning}
This setup assumes that all data is represented as real numbers (since both $x^{(i)}$ and $y^{(i)}$ are real-valued vectors). Handling non-numeric data (e.g. text, categories) requires a pre-processing step named *encoding* (or *embedding*), which will be briefly discussed in the following.
```

### Mathematical Formulation of Machine Learning Models

At its core, a Machine Learning (ML) model is a function

$$
f_\Theta: \mathbb{R}^d \to \mathbb{R}^s
$$

that maps an input vector of length $d$ to an output vector of length $s$. The model's knowledge is encoded in the vector $\Theta$, which represents its **parameters**. To better understand the role of parameters in an ML model, let's start with the simplest possible example: **Linear Regression**.

Consider a basic model $f_\Theta$ with just two parameters, $\Theta_1$ and $\Theta_2$:

$$
f_\Theta(x) = \Theta_1 + \Theta_2 x.
$$

This equation describes a straight line where $\Theta_1$ is the intercept and $\Theta_2$ is the slope.

![Predictions of a linear model for two choices of parameters.](./fig/linear_model_parameters.png)

Think of $x$ as the input data that we use to predict an output $y$. Notice how different choices of $\Theta$ can lead to vastly different predictions.

Now, let's overlay the available data on the same plot:

![Two linear models compared with the observed data.](./fig/linear_model_fit.png)

*Which of the two straight lines better fits the given data?*
Clearly, the blue line, parameterized by $\Theta=(1,0.2)$. The data were generated as $y^{(i)} = 1 + 0.2 x^{(i)} + \varepsilon^{(i)}$, with Gaussian noise $\varepsilon^{(i)} \sim \mathcal{N}(0, 0.5^2)$.

This highlights two fundamental characteristics common to almost all ML models:

* Different choices of $\Theta$ lead to *very different* predictions.
* Given a training set, some parameter choices are better than others, and usually, there is one that is *optimal*. A good model should at least approximate this optimal choice.

The process of finding the optimal parameters for an ML model based on a training set is called **training**. As we will see, training is nothing else than an **optimization problem**, and solving it is the main topic of this course.

### Linearity

In the example above, we assumed both the input and output dimensions were $d = 1$ and $s = 1$. However, we can easily generalize a linear regression model to higher dimensions ($d > 1$, $s > 1$) using the general linear model:

$$
f_\Theta(x) = W x + b
$$

where the parameters $\Theta = \{ W, b \}$ have dimensions $W \in \mathbb{R}^{s \times d}$ and $b \in \mathbb{R}^s$. This means the total number of parameters to be learned is $s(d+1)$, which grows linearly with the input and output dimensions.

While linear models have several useful properties, their **expressivity** (i.e., their ability to approximate complex input-output relationships) is quite limited. A linear model can only represent *linear functions* (such as straight lines and planes), making it incapable of even approximating simple functions like $\sin(x)$.

![A linear model cannot capture the oscillations of sine.](./fig/linear_model_limitations.png)

This limitation makes linear models impractical for handling complex data, such as images or natural language. For this reason, multiple non-linear ML algorithms have been proposed in the literature, such as Polynomial Regression, Support Vector Machines (SVM), Random Forest (RF), XGBoost and, of course, **Neural Networks**, which will be the subject of the next chapter.

## Types of Learning

Designing a Machine Learning model requires choosing the right *paradigm* depending on the type of problem we want to solve and, most importantly, on the type of data we have at disposal. Broadly speaking, learning algorithms can be grouped into three main categories:

- **Supervised Learning**
- **Unsupervised Learning**
- **Self-Supervised Learning** (advanced, not covered in this course)

![](./fig/diagram.png)

### Supervised Learning

In **Supervised Learning (SL)**, we are given a dataset composed of *input–output* pairs:

$$
\mathcal{D} = \{ (x^{(i)}, y^{(i)}) \}_{i=1}^N,
$$

where $x^{(i)} \in \mathbb{R}^d$ is the input (a feature vector) and $y^{(i)}$ is the corresponding **label** (or *target*). The labels are usually obtained by a human (or by a measurement process), who acts as a *supervisor*, from which the name.

The idea of supervised learning is to learn a function:

$$
f_\Theta : \mathbb{R}^d \to \mathcal{Y},
$$

parameterized by $\Theta$, such that $f_\Theta(x^{(i)}) \approx y^{(i)}$ for any $i = 1, \dots, N$ and, more importantly, such that $f_\Theta(x) \approx y$ also for **new** inputs $x$ not contained in the dataset.

Supervised learning can be further divided into two major tasks, depending on the nature of the output space $\mathcal{Y}$:

- **Regression**: The output $y^{(i)}$ is a real-valued quantity, i.e., $y^{(i)} \in \mathbb{R}^s$.
  Example: predicting house prices based on property features, predicting tomorrow's temperature.

- **Classification**: The output $y^{(i)}$ belongs to a *finite* set of classes, i.e., $y^{(i)} \in \{1, \ldots, K\}$.
  Example: classifying an email as *spam* or *not spam*, recognizing which digit is written in an image.

![](./fig/regression_vs_classification.png)

The difference between the two tasks is subtle but fundamental. In regression, the model output is a *number* (or a vector of numbers), and an error of $0.1$ is "better" than an error of $10$. In classification, the model output is a *decision*: the prediction is either correct or wrong. Geometrically, a classifier divides the input space $\mathbb{R}^d$ into $K$ regions, one per class, and the boundaries between those regions are named **decision boundaries**.

![](./fig/classification_2.png)

```{note}
From the mathematical point of view, it is hard to optimize a "correct/wrong" quantity, since it is piecewise constant (and therefore its gradient is zero almost everywhere). For this reason, in practice, a classifier rarely outputs the class directly. Instead, it outputs a **probability** for each class:

$$
f_\Theta(x) = \big( \mathbb{P}(y = 1 \mid x), \mathbb{P}(y = 2 \mid x), \dots, \mathbb{P}(y = K \mid x) \big) \in [0, 1]^K,
$$

and the predicted class is the one with the largest probability. In this way, a classification problem becomes a (very particular) regression problem with output dimension $s = K$.

In this course, we will focus on **binary classification**, i.e. $K = 2$ and $y \in \{0, 1\}$. In this case, it is sufficient to output a **single** number, $f_\Theta(x) = \mathbb{P}(y = 1 \mid x) \in [0, 1]$, since $\mathbb{P}(y = 0 \mid x) = 1 - f_\Theta(x)$. We will come back on this in the next chapter.
```

In the following code, we visualize a synthetic regression dataset and a synthetic (binary) classification dataset.

![Regression predicts a real value; classification predicts a discrete label.](./fig/regression_classification_examples.png)

Note that, in the regression case, the input $x \in \mathbb{R}$ is represented on the horizontal axis while the output $y$ on the vertical axis. In the classification case, the input $x = (x_1, x_2) \in \mathbb{R}^2$ is represented by the position of the point in the plane, while the output (the class) is represented by its **color**. Clearly, no straight line can separate the two classes in the right plot: we will need a *non-linear* model.

### Unsupervised Learning

In **Unsupervised Learning (UL)**, we are only given input data without labels:

$$
\mathcal{D} = \{ x^{(i)} \}_{i=1}^N, \quad x^{(i)} \in \mathbb{R}^d.
$$

Here, the task is not to predict a known output, but rather to uncover **hidden structures or patterns** in the data. The model attempts to organize, compress, or explain the data in some meaningful way.

Common tasks include:

- **Clustering**: Grouping data points into clusters such that points in the same group are more similar to each other than to points in other groups.
  Example: market segmentation of customers based on their purchasing behavior.

- **Dimensionality Reduction**: Mapping high-dimensional data into a lower-dimensional space while preserving as much information as possible.
  Example: Principal Component Analysis (PCA) for data visualization.

Unsupervised learning is often used when labeled data is expensive or impossible to obtain, making it a powerful tool in exploratory data analysis.

### Self-Supervised Learning

**Self-Supervised Learning (SSL)** is a paradigm that sits between supervised and unsupervised learning. Unlike supervised learning, no human-annotated labels are required. Instead, the model creates its own labels by solving *pretext tasks* defined directly on the input data. Examples of pretext tasks include:

- Predicting the next word in a sentence (used to train Large Language Models such as GPT or Claude).
- Predicting missing parts of an image or the next video frame.
- Contrastive learning, where the model learns by comparing different views of the same data.

The key idea is that by solving these automatically generated tasks, the model learns **useful representations** of data, which can then be fine-tuned on smaller labeled datasets for supervised tasks.

In summary:

- **Supervised Learning**: learn from labeled data ($x, y$).
- **Unsupervised Learning**: learn from unlabeled data ($x$).
- **Self-Supervised Learning**: generate labels from data itself to pre-train representations.

In this course, we will focus on **supervised classification**, which is the setting where Neural Networks are the easiest to introduce.

## The Machine Learning Pipeline

The examples above show that designing an effective Machine Learning algorithm requires several careful choices, such as the structure of the model, the amount and quality of data, and the values of the parameters. In particular, we can identify the following steps, which are common to most ML algorithms:

- **Understanding** the task (e.g., what do we need? what information can we collect to answer the question we are asking?),
- **Collecting** a sufficiently large dataset, containing enough relevant information to address the task,
- **Designing** the Machine Learning algorithm, leveraging prior knowledge about the problem under study,
- **Training** the algorithm on the collected data, by minimizing the prediction error on the training set,
- **Tuning** the hyperparameters of the model (in ML, an algorithm is often referred to as a *model*) to improve its predictions,
- **Testing** the algorithm on new data, in order to verify its predictive ability.

We will briefly examine each of these steps in the following.

### Understanding the task
Assume we want to solve a given problem. Mathematically, the problem we aim to solve can be modelled as an (unknown) function $f(x)$, taking as input the vector $x \in \mathbb{R}^d$, and mapping it (possibly **stochastically**) to the task $y = f(x)$.

A Machine Learning model $f_\Theta (x)$, parameterized by $\Theta$, is said to **solve** the problem if there exists a choice of parameters $\Theta$ such that

$$
f_\Theta(x) \approx f(x) \qquad \forall x \in \mathbb{R}^d.
$$

#### Is it learnable?
A problem $y = f(x)$ can be solved by a ML algorithm if and only if there **exists** a relationship between $x$ and $y$. For example, we cannot expect to predict the weather in a particular position by using information about the stock price of a particular company. In that situation, the input and the output are clearly **independent**, and there is no chance of learning anything from one using the other.

Consequently, the first point in designing a ML algorithm is to understand _if_ there exists a correlation between the input and the output of the given problem. When this is the case, we say that the problem is **learnable**.

> **Machine Learning** is about understanding correlations (patterns).

#### Is it possible to collect data?
Assume that the problem $y = f(x)$ is learnable. We need to understand if we can physically collect enough data $x$ to be able to understand the relationship between it and its corresponding $y$.

For example, if we want to use ML to make cancer diagnosis on patients, clearly the best way to do that is to use as input the clinical results of any possible medical exam on the patient. Of course, even if this would work well in practice, it is not possible (and especially not ethical) to test the patient with thousands of exams for a single diagnosis.

Moreover, to train a good ML model, we will need thousands (sometimes millions) of datapoints, and it is not always possible to scale our problem to be able to collect enough data to solve it.

> Collecting data requires **efficiency** and **scalability** of the problem.

### Collecting data
Collecting data is usually the step that takes the most effort in the design of a Machine Learning algorithm. In fact, given that our problem $y = f(x)$ is solvable and that it is **theoretically possible** to collect enough data about it, it is not always that **easy** in practice.

In particular, some data requires *time* to be collected (this is the case when working in biological or medical applications), and collecting good quality data is hard. Indeed, we ideally want to use a clean dataset, where all the information is present, there are no missing values (usually referred to as `NaN`) and the data does not contain noise. Most of the time, this is hopeless, and we will need to develop algorithms to standardize and clean up the data. The set of all those techniques is called **data cleaning**.

Luckily, for most of the tasks you can think of, you can find datasets on the internet. For example, websites like [Kaggle](https://www.kaggle.com/), [Hugging Face Datasets](https://huggingface.co/datasets) and [Google Datasets](https://datasetsearch.research.google.com/) can be helpful for that. In the last chapter of this course, we will download one of these datasets and we will train a Neural Network on it.

#### Datasets as matrices

Datasets are usually distributed in formats such as `.csv` (tables), or as collections of files (e.g. images). However, from a mathematical and computational perspective, Machine Learning algorithms expect data in the form of **numeric vectors and matrices**. For this reason, we almost always **convert datasets into matrices**, typically represented as `numpy` arrays. In this course, we will always use the convention that the dataset is stored as a matrix

$$
X = \begin{bmatrix} (x^{(1)})^T \\ (x^{(2)})^T \\ \vdots \\ (x^{(N)})^T \end{bmatrix} \in \mathbb{R}^{N \times d},
$$

where each **row** $x^{(i)} \in \mathbb{R}^d$ is a datapoint with $d$ features, and $N$ is the number of samples. The corresponding labels are collected in a vector $Y = [y^{(1)}, y^{(2)}, \ldots, y^{(N)}]^T \in \mathbb{R}^N$ (or in a matrix $Y \in \mathbb{R}^{N \times s}$ if $s > 1$).

When some features are **categorical** (e.g. *weather* $\in \{ \text{sunny}, \text{rainy}, \text{cloudy}, \text{snowy} \}$), they have to be converted to numbers. The simplest strategy is the **One-Hot Encoding**: each class $C_k$ is mapped to the $k$-th canonical vector $e_k \in \mathbb{R}^K$, which is all zeros except for a single $1$ at position $k$. For example, with $K = 4$ classes:

$$
\text{sunny} \mapsto \begin{bmatrix} 1 \\ 0 \\ 0 \\ 0 \end{bmatrix}, \quad
\text{rainy} \mapsto \begin{bmatrix} 0 \\ 1 \\ 0 \\ 0 \end{bmatrix}, \quad
\text{cloudy} \mapsto \begin{bmatrix} 0 \\ 0 \\ 1 \\ 0 \end{bmatrix}, \quad
\text{snowy} \mapsto \begin{bmatrix} 0 \\ 0 \\ 0 \\ 1 \end{bmatrix}.
$$

In `numpy`, one-hot encoding can be obtained with a single line of code:

In [1]:
import numpy as np

# Labels of 6 datapoints, with K = 4 classes
y = np.array([0, 2, 1, 3, 2, 0])
K = 4

# Row k of the identity matrix is exactly e_k
Y_onehot = np.eye(K)[y]
print(Y_onehot)

[[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]]


```{note}
Modern models (in particular Large Language Models) do not use one-hot encoding for their inputs, as it does not capture **semantic relationships** between categories (in one-hot encoding, *sunny* and *cloudy* are orthogonal vectors, even though they are more similar to each other than to *snowy*). Instead, they use **learned embeddings**: each category (e.g. each word) is mapped to a dense vector in $\mathbb{R}^d$, which is learned jointly with the parameters of the model.
```

### Design

Designing the model means choosing the family of functions $f_\Theta$ among which we look for the solution. This choice depends on the type of task (regression, classification, clustering, ...), on the type of data (tabular data, images, text, ...) and on the prior knowledge we have about the problem. For example, if we know that the relationship between $x$ and $y$ is approximately linear, a linear model would be enough, while for images we need very expressive models such as Neural Networks.

The design of a Neural Network classifier will be the topic of the next chapter.

### Training

The **training phase** is at the heart of every Machine Learning algorithm. Once we have defined a model and collected data, training is the process by which the model *learns* the best parameters $\Theta$ to approximate the underlying function we are trying to capture.

Training requires a way to measure **how good** a model’s predictions are. This is done through a **loss function** (also called a cost function or objective function), denoted as:

$$
\mathcal{L}(\Theta) = \frac{1}{N} \sum_{i=1}^N \ell(f_\Theta(x^{(i)}), y^{(i)}),
$$

where:
- $(x^{(i)}, y^{(i)})$ are the datapoints from the training set,
- $f_\Theta(x^{(i)})$ is the model prediction,
- $\ell(\cdot, \cdot)$ is a *local* error measure, comparing the prediction with the true label.

For example, in regression, a common choice is the **Squared Error**:

$$
\ell(f_\Theta(x), y) = \| f_\Theta(x) - y \|_2^2,
$$

which, averaged over the dataset, gives the well-known **Mean Squared Error (MSE)**. For classification, the standard choice is the **Cross-Entropy**, which we will introduce in the next chapter.

The **training problem** is then the optimization problem:

$$
\Theta^* = \arg\min_\Theta \mathcal{L}(\Theta).
$$

```{important}
From a mathematical perspective, **training a Machine Learning model is equivalent to solving an optimization problem**. In most practical settings, this problem cannot be solved analytically:
- the loss is usually non-linear (and non-convex) in the parameters,
- it may involve millions (or even billions) of parameters,
- the dataset size $N$ can be extremely large.

For this reason, we need **iterative optimization algorithms**, such as Gradient Descent (Chapter 3), Stochastic Gradient Descent (Chapter 4) and Adam (Chapter 5).
```

### Tuning

Most Machine Learning models have a number of **hyperparameters** that are not directly learned during training but must be set by the practitioner. Examples include the learning rate in Gradient Descent, the number of layers of a Neural Network, or the number of clusters in a clustering algorithm.

Tuning consists of systematically adjusting these hyperparameters to improve the model’s performance. This is usually done through a **trial-and-error process**, guided by prior knowledge of the model and by evaluating performance on a separate **validation set**.

### Testing

Once a model has been trained (parameters optimized) and tuned (hyperparameters chosen), we need to evaluate its **ability to generalize**. This means assessing how well the model performs on data it has never seen before.

Using the same dataset for both training and testing is misleading. On training data, the model has already observed the true outcomes, so it might simply **memorize** the data. To avoid this, the dataset is typically split into:
- **Training set**: used to adjust model parameters,
- **Validation set**: used to tune hyperparameters,
- **Test set**: used only once at the end, to provide an unbiased estimate of performance.

![](./fig/train_test_split.png)

The choice of metric to evaluate the model depends on the task. For regression, the MSE (or the Mean Absolute Error) on the test set is the most common choice. For classification, the most natural metric is the **accuracy**, i.e. the proportion of correctly classified samples:

$$
\text{Acc} = \frac{1}{N_{\text{test}}} \sum_{i=1}^{N_{\text{test}}} \mathbb{1}\{\hat{y}^{(i)} = y^{(i)}\},
$$

where $\hat{y}^{(i)}$ is the predicted class of the $i$-th test sample.

#### Underfitting vs Overfitting

A crucial aspect of testing is understanding the **generalization behavior** of a model. Two extreme situations can occur:

- **Underfitting**: the model is *too simple*. It fails to capture the underlying structure of the data, leading to **high error** both on the training set and on the test set (e.g. a straight line trying to fit the $\sin(x)$ function above).

- **Overfitting**: the model is *too complex*. It adapts too closely to the training data, capturing even noise and irrelevant patterns. This leads to **low error** on the training set but **high error** on the test set.

![](./fig/overfit_underfit.png)

The ideal model lies **between underfitting and overfitting**, achieving low training error and comparable test error. The difference between these two errors is called the **generalization gap**.

## Summary

- **AI** $\supset$ **ML** $\supset$ **DL**: Machine Learning is the part of AI that learns from data, Deep Learning is the part of ML based on Neural Networks.
- A ML model is a parametric function $f_\Theta: \mathbb{R}^d \to \mathbb{R}^s$.
- In **supervised learning** we have labels: **regression** if $y$ is continuous, **classification** if $y$ belongs to a finite set of classes.
- **Training** a model means solving the optimization problem $\min_\Theta \mathcal{L}(\Theta)$.
- The model must be evaluated on data that was **not** used for training.

In the next chapter, we will design our first non-linear classifier: a **Neural Network**.

> **Exercise:** For each of the following problems, say whether it is a regression or a classification problem, and describe what $x$ and $y$ are (with their dimensions $d$ and $s$):
> 1. Predicting the age of a person from a $64 \times 64$ grayscale photo of their face.
> 2. Recognizing whether a $28 \times 28$ grayscale image contains a handwritten digit $0, 1, \dots, 9$.
> 3. Predicting the price of 3 different stocks tomorrow, given their price in the last 30 days.
> 4. Deciding whether a credit card transaction is fraudulent, given its amount, time and location.